# 06 — Strong 2026 baseline: intermediate-layer visual targets (Shallow Alignment / SAMGA recipe) + label-free recovery

Goal: reproduce the 2026 state of the art on THINGS-EEG2 before adding anything of our own.

| Part | What | Reference numbers (papers, Top-1, 200-way) |
|---|---|---|
| Features | InternViT-6B-448px-V2.5, layers 20/24/28/32/36, frozen (fallback: DINOv2-giant) | — |
| Within-subject | EEGProject (MLP) + linear projectors; single layer sweep and learned layer routing; 17 posterior channels | Shallow Alignment 82.6, SAMGA 91.3 |
| LOSO, 10 folds | same, 63 channels, train on 9 subjects | Shallow Alignment 22.4, SAMGA 34.4 |
| Label-free recovery | CSLS and SCORE-style coordinate recovery on the LOSO model (transductive, no labels of the new person) | SCORE on SAMGA 26.2 → 48.8 (recovery) |

Settings follow SAMGA Table 1: 60 epochs, AdamW, batch 1024, lr 1e-4, weight decay 1e-4, routing logits initialised [-2,-1,0,-1,-2].
The best epoch is chosen on held-out training images (never on the test set).

Uses the Drive cache from notebooks 04/05 (`*_z16.npy`, image list). Every result is saved as JSON; a re-run skips finished parts.

**GPU:** A100 is best for cell 2 (InternViT-6B features, ~15–25 min, once). L4 works (~45–60 min). Cells 5–6 run fine on L4.

In [1]:
# 0. Setup
import os, sys, gc, json, math, time, random, shutil, glob, copy, importlib.util, subprocess
try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    pass
DRIVE  = os.environ.get("EEG_DRIVE", "/content/drive/MyDrive/eeg_results")
DCACHE = f"{DRIVE}/cache"
ROOT   = os.environ.get("EEG_ROOT", "/content/EEG_Data")
LOCAL  = os.environ.get("EEG_LOCAL", "/content/cache")
OUT    = f"{DRIVE}/v6"
for d in [DRIVE, DCACHE, ROOT, LOCAL, OUT]: os.makedirs(d, exist_ok=True)
ALL_SUBJECTS = [f"sub-{i:02d}" for i in range(1, 11)]
SMOKE = os.environ.get("EEG_SMOKE") == "1"           # tiny synthetic run for code checking only

import numpy as np
for pkg, pip_name in [("huggingface_hub", "huggingface_hub"), ("pandas", "pandas"), ("transformers", "transformers"),
                      ("timm", "timm"), ("einops", "einops")]:
    if not SMOKE and importlib.util.find_spec(pkg) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pip_name], check=True)
import pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
device = "cuda" if torch.cuda.is_available() else "cpu"
AMP = device == "cuda" and torch.cuda.get_device_capability()[0] >= 8      # bfloat16 needs Ampere or newer (not T4)
print("device:", device, torch.cuda.get_device_name(0) if device == "cuda" else "", "| bf16:", AMP, "| torch", torch.__version__)
print(f"Drive free: {shutil.disk_usage(DRIVE).free / 1e9:.1f} GB (features need ~1.2 GB)")

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)

def local_copy(name):
    dst = f"{LOCAL}/{name}"
    if not os.path.exists(dst): shutil.copy(f"{DCACHE}/{name}", dst + ".tmp"); os.replace(dst + ".tmp", dst)
    return dst

def save_json(obj, path):
    json.dump(obj, open(path + ".tmp", "w")); os.replace(path + ".tmp", path)

def free_gpu():
    gc.collect()
    if device == "cuda": torch.cuda.empty_cache()

Mounted at /content/drive
device: cuda NVIDIA A100-SXM4-40GB | bf16: True | torch 2.11.0+cu130
Drive free: 4.7 GB (features need ~1.2 GB)


In [2]:
# 1. Image list (same order as notebooks 04/05) and the validation split
META = f"{DCACHE}/meta_v6.pt"
if not os.path.exists(META):
    import tarfile
    from huggingface_hub import hf_hub_download
    if not os.path.exists(f"{ROOT}/images_set/image_metadata.npy"):
        tar_path = hf_hub_download(repo_id="LidongYang/EEG_Image_decode", filename="images_set.tar.gz", repo_type="dataset", local_dir=ROOT)
        with tarfile.open(tar_path, "r:gz") as t: t.extractall(path=ROOT)
    obj = np.load(f"{ROOT}/images_set/image_metadata.npy", allow_pickle=True)
    meta = obj.item() if isinstance(obj, np.ndarray) and obj.dtype == object and obj.ndim == 0 else obj
    train_files, test_files = list(meta["train_img_files"]), list(meta["test_img_files"])
    def build_index(folder):
        idx = {}
        for dp, _, fs in os.walk(folder):
            for f in fs: idx.setdefault(f, os.path.join(dp, f))
        return idx
    img_root = f"{ROOT}/images_set"
    idx_train = build_index(f"{img_root}/training_images")
    idx_test = build_index(f"{img_root}/test_images") if os.path.isdir(f"{img_root}/test_images") else build_index(img_root)
    torch.save({"train_files": train_files, "test_files": test_files,
                "train_rel": [os.path.relpath(idx_train[f], ROOT) for f in train_files],
                "test_rel": [os.path.relpath(idx_test[f], ROOT) for f in test_files],
                "n_concepts": len(set(meta["train_img_concepts"]))}, META)
M = torch.load(META, weights_only=False)
N_TR, N_TE = len(M["train_files"]), len(M["test_files"])
PER = N_TR // M["n_concepts"]
IS_VAL = (np.arange(N_TR) % PER) == (PER - 1)              # last image of every training concept (same as 01–05)
print(f"train images {N_TR} | test images {N_TE} | concepts {M['n_concepts']} | val images {IS_VAL.sum()}")

images_set.tar.gz: reconstructing file:   0%|          |  0.00B /  778MB            

images_set.tar.gz: downloading bytes:           |  0.00B            

/tmp/ipykernel_823/3031348737.py:8: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  with tarfile.open(tar_path, "r:gz") as t: t.extractall(path=ROOT)


train images 16540 | test images 200 | concepts 1654 | val images 1654


In [3]:
# 2. Intermediate-layer image features, cached on Drive (one file per backbone, all layers and two poolings)
#    InternViT-6B-448px-V2.5 is the SAMGA backbone; DINOv2-giant is the fallback if it cannot be loaded.
BACKBONES = {
    "internvit6b": {"repo": "OpenGVLab/InternViT-6B-448px-V2_5", "layers": [20, 24, 28, 32, 36]},
    "dinov2g":     {"repo": "facebook/dinov2-giant",             "layers": [16, 20, 24, 28, 32, 36]},
}
BACKBONE = os.environ.get("EEG_BACKBONE", "internvit6b")
FEAT = lambda b: f"{DCACHE}/feat6_{b}.pt"

def extract(bname):
    from PIL import Image
    from torch.utils.data import Dataset, DataLoader
    from transformers import AutoModel, AutoImageProcessor
    cfg = BACKBONES[bname]; t0 = time.time()
    if not os.path.exists(f"{ROOT}/images_set/image_metadata.npy"):
        import tarfile
        from huggingface_hub import hf_hub_download
        tar_path = hf_hub_download(repo_id="LidongYang/EEG_Image_decode", filename="images_set.tar.gz", repo_type="dataset", local_dir=ROOT)
        with tarfile.open(tar_path, "r:gz") as t: t.extractall(path=ROOT)
    dt = torch.bfloat16 if AMP else torch.float16
    if bname == "internvit6b":
        from transformers import CLIPImageProcessor
        proc = CLIPImageProcessor.from_pretrained(cfg["repo"])
        model = AutoModel.from_pretrained(cfg["repo"], torch_dtype=dt, low_cpu_mem_usage=True, trust_remote_code=True,
                                          use_flash_attn=importlib.util.find_spec("flash_attn") is not None)
        model.encoder.layers = model.encoder.layers[:max(cfg["layers"])]      # deeper blocks are never used
    else:
        proc = AutoImageProcessor.from_pretrained(cfg["repo"])
        model = AutoModel.from_pretrained(cfg["repo"], torch_dtype=dt)
        model.encoder.layer = model.encoder.layer[:max(cfg["layers"])]
    model = model.to(device).eval()
    print(f"{bname} loaded in {(time.time()-t0)/60:.1f} min", flush=True)

    class DS(Dataset):
        def __init__(self, rel): self.rel = rel
        def __len__(self): return len(self.rel)
        def __getitem__(self, i): return Image.open(f"{ROOT}/{self.rel[i]}").convert("RGB")
    def collate(ims): return proc(images=ims, return_tensors="pt")["pixel_values"]

    def run(rel, tag):
        out = {(l, p): [] for l in cfg["layers"] for p in ("mean", "cls")}
        n = 0
        for px in DataLoader(DS(rel), batch_size=16, num_workers=4, collate_fn=collate):
            with torch.no_grad():
                hs = model(pixel_values=px.to(device, dt), output_hidden_states=True).hidden_states
            for l in cfg["layers"]:
                h = hs[l].float()
                out[(l, "cls")].append(h[:, 0].cpu().half()); out[(l, "mean")].append(h[:, 1:].mean(1).cpu().half())
            n += len(px)
            if n % 1600 < 16: print(f"  {tag} {n}/{len(rel)} | {(time.time()-t0)/60:.1f} min", flush=True)
        return {k: torch.cat(v) for k, v in out.items()}
    tr, te = run(M["train_rel"], "train"), run(M["test_rel"], "test")
    d = {"layers": cfg["layers"], "tr": tr, "te": te}
    torch.save(d, FEAT(bname) + ".tmp"); os.replace(FEAT(bname) + ".tmp", FEAT(bname))
    del model; free_gpu()
    print(f"{bname} features done in {(time.time()-t0)/60:.1f} min")

if not os.path.exists(FEAT(BACKBONE)):
    try:
        extract(BACKBONE)
    except Exception as e:
        print(f"{BACKBONE} failed: {type(e).__name__}: {str(e)[:300]}\n-> falling back to dinov2g")
        BACKBONE = "dinov2g"
        if not os.path.exists(FEAT(BACKBONE)): extract(BACKBONE)
FE = torch.load(FEAT(BACKBONE), weights_only=False)
LAYERS = FE["layers"]
print("backbone:", BACKBONE, "| layers:", LAYERS, "| dim:", FE["tr"][(LAYERS[0], "mean")].shape[1])

# old target from notebook 04 for comparison (CLIP ViT-H/14 final embedding)
F04 = f"{DCACHE}/feats3_ViT-H-14_laion2b_s32b_b79k.pt"
H14 = torch.load(F04, weights_only=False) if os.path.exists(F04) else None

preprocessor_config.json:   0%|          | 0.00/287 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/801 [00:00<?, ?B/s]

configuration_intern_vit.py:   0%|          | 0.00/5.48k [00:00<?, ?B/s]

[transformers] A new version of the following files was downloaded from https://huggingface.co/OpenGVLab/InternViT-6B-448px-V2_5:
- configuration_intern_vit.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


modeling_intern_vit.py:   0%|          | 0.00/14.0k [00:00<?, ?B/s]

flash_attention.py:   0%|          | 0.00/3.37k [00:00<?, ?B/s]

... (4 outputs left out)

model.safetensors: reconstructing file:   0%|          |  0.00B / 4.55GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/727 [00:00<?, ?it/s]

dinov2g loaded in 0.3 min
  train 1600/16540 | 0.4 min
  train 3200/16540 | 0.5 min
  train 4800/16540 | 0.6 min
  train 6400/16540 | 0.7 min
  train 8000/16540 | 0.8 min
  train 9600/16540 | 0.9 min
  train 11200/16540 | 1.0 min
  train 12800/16540 | 1.2 min
  train 14400/16540 | 1.3 min
  train 16000/16540 | 1.4 min
dinov2g features done in 1.4 min
backbone: dinov2g | layers: [16, 20, 24, 28, 32, 36] | dim: 1536


In [4]:
# 3. Targets: per-dimension z-score with training statistics (intermediate ViT layers have a few huge-norm dims)
def target(name, pool="mean"):
    if name == "h14final":
        tr, te = H14["img_tr"].float(), H14["img_te"].float()
    else:
        tr, te = FE["tr"][(name, pool)].float(), FE["te"][(name, pool)].float()
    m, s = tr.mean(0, keepdim=True), tr.std(0, keepdim=True) + 1e-4
    return ((tr - m) / s).to(device), ((te - m) / s).to(device)

# EEG channels. THINGS-EEG2 preprocessing order (Gifford et al. 2022); the 17 posterior channels used by NICE/UBP/SAMGA.
CH63 = ["Fp1","Fp2","AF7","AF3","AFz","AF4","AF8","F7","F5","F3","F1","F2","F4","F6","F8","FT9","FT7","FC5","FC3","FC1",
        "FCz","FC2","FC4","FC6","FT8","FT10","T7","C5","C3","C1","Cz","C2","C4","C6","T8","TP9","TP7","CP5","CP3","CP1",
        "CPz","CP2","CP4","CP6","TP8","TP10","P7","P5","P3","P1","Pz","P2","P4","P6","P8","PO7","PO3","POz","PO4","PO8",
        "O1","Oz","O2"]
POST17 = ["P7","P5","P3","P1","Pz","P2","P4","P6","P8","PO7","PO3","POz","PO4","PO8","O1","Oz","O2"]
assert len(CH63) == 63
_, N_CH_ALL, N_T = np.load(f"{DCACHE}/sub-01_test_z16.npy", mmap_mode="r").shape
CH_IDX = {"all": list(range(N_CH_ALL)),
          "post17": [CH63.index(c) for c in POST17] if N_CH_ALL == 63 else list(range(N_CH_ALL))}
print("EEG:", N_CH_ALL, "channels x", N_T, "samples | post17 indices", CH_IDX["post17"][:3], "...")

def load_eeg(sub, split, ch):
    x = np.load(local_copy(f"{sub}_{split}_z16.npy"), mmap_mode="r")
    return torch.from_numpy(np.ascontiguousarray(x[:, CH_IDX[ch]])).to(device)

VAL = torch.from_numpy(np.where(IS_VAL)[0]).to(device); TRN = torch.from_numpy(np.where(~IS_VAL)[0]).to(device)

EEG: 63 channels x 250 samples | post17 indices [46, 47, 48] ...


In [5]:
# 4. Model: EEGProject (MLP with residual bottleneck, as in Shallow Alignment) + linear image projector(s)
#    Single-layer target, or a learned softmax routing over several layers (global SAMGA-style prior, no subject residual).
EMB = 1024

class EEGProject(nn.Module):
    def __init__(self, n_in, d=EMB, drop=0.3):
        super().__init__()
        self.inp = nn.Linear(n_in, d)
        self.res = nn.Sequential(nn.GELU(), nn.Linear(d, d), nn.Dropout(drop))
        self.norm = nn.LayerNorm(d)
    def forward(self, x):
        h = self.inp(x.flatten(1)); return self.norm(h + self.res(h))

class ImageSide(nn.Module):
    def __init__(self, dims, init_logits=None):
        super().__init__()
        self.proj = nn.ModuleList([nn.Linear(d, EMB) for d in dims])
        self.logits = nn.Parameter(torch.tensor(init_logits, dtype=torch.float) if init_logits is not None else torch.zeros(len(dims)),
                                   requires_grad=len(dims) > 1)
    def weights(self): return F.softmax(self.logits, 0)
    def forward(self, feats):                   # feats: list of (N, d_l), same rows
        w = self.weights()
        return sum(w[i] * p(f) for i, (p, f) in enumerate(zip(self.proj, feats)))

class Aligner(nn.Module):
    def __init__(self, n_in, dims, init_logits=None):
        super().__init__()
        self.eeg, self.img = EEGProject(n_in), ImageSide(dims, init_logits)
        self.log_scale = nn.Parameter(torch.tensor(math.log(1 / 0.07)))
    def scale(self): return self.log_scale.exp().clamp(max=100)

def clip_loss(ze, zi, scale):
    ze, zi = F.normalize(ze, dim=-1), F.normalize(zi, dim=-1)
    logits = scale * ze @ zi.T; y = torch.arange(len(ze), device=ze.device)
    return (F.cross_entropy(logits, y) + F.cross_entropy(logits.T, y)) / 2

@torch.no_grad()
def embed_eeg(model, X, bs=4096):
    model.eval(); out = []
    for i in range(0, len(X), bs):
        with torch.autocast(device_type="cuda", dtype=torch.bfloat16, enabled=AMP):
            out.append(F.normalize(model.eeg(X[i:i + bs].float()).float(), dim=-1))
    return torch.cat(out)

@torch.no_grad()
def embed_img(model, feats):
    model.eval(); return F.normalize(model.img(feats).float(), dim=-1)

@torch.no_grad()
def metrics(sim):
    # sim: queries x candidates, the correct candidate of query i is candidate i
    r = (sim > sim.diag().unsqueeze(1)).sum(1)
    return {"top1": (r < 1).float().mean().item() * 100, "top5": (r < 5).float().mean().item() * 100}

EPOCHS, BATCH, LR, WD, EVAL_EVERY = (60, 1024, 1e-4, 1e-4, 5) if not SMOKE else (3, 64, 1e-3, 1e-4, 1)
ROUTE_INIT = [-2.0, -1.0, 0.0, -1.0, -2.0]

def train(X, img_of, T_tr, val_sets, seed=0, log=None, init_logits=None):
    # X: EEG trials, img_of[i]: training image of trial i, T_tr: list of target matrices (N_TR x d_l)
    # val_sets: list of (EEG rows, image rows) used for model selection (held-out training images)
    set_seed(seed)
    model = Aligner(X[0].numel(), [t.shape[1] for t in T_tr], init_logits).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WD)
    tr_rows = torch.where(~torch.from_numpy(IS_VAL).to(device)[img_of])[0]
    steps = EPOCHS * math.ceil(len(tr_rows) / BATCH)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=LR, total_steps=steps, pct_start=0.05)
    best, best_state, best_ep = -1, None, 0
    for ep in range(1, EPOCHS + 1):
        model.train(); perm = tr_rows[torch.randperm(len(tr_rows), device=device)]
        for i in range(0, len(perm), BATCH):
            b = perm[i:i + BATCH]
            if len(b) < 2: continue
            imgs = img_of[b]
            with torch.autocast(device_type="cuda", dtype=torch.bfloat16, enabled=AMP):
                ze = model.eeg(X[b].float()); zi = model.img([t[imgs] for t in T_tr])
            loss = clip_loss(ze.float(), zi.float(), model.scale())
            opt.zero_grad(set_to_none=True); loss.backward(); opt.step(); sched.step()
        if ep % EVAL_EVERY == 0 or ep == EPOCHS:
            zi_val = embed_img(model, [t[VAL] for t in T_tr])
            va = float(np.mean([metrics(embed_eeg(model, X[er]) @ zi_val[ir].T)["top1"] for er, ir in val_sets]))
            if log: log(ep, va, model)
            if va > best: best, best_ep, best_state = va, ep, copy.deepcopy(model.state_dict())
    model.load_state_dict(best_state)
    return model, best_ep, best

In [6]:
# 5. Within-subject, all 10 subjects. Ordered by priority; each (config, subject) is saved and skipped on re-run.
#    name: list of layers (or "h14final"), pooling, channels
CONFIGS = [("route", LAYERS[-5:], "mean", "post17"),
           ("h14final", ["h14final"], "mean", "post17")] + \
          [(f"L{l}", [l], "mean", "post17") for l in LAYERS] + \
          [(f"L{LAYERS[-3]}cls", [LAYERS[-3]], "cls", "post17"),
           ("route_ch63", LAYERS[-5:], "mean", "all")]
if H14 is None: CONFIGS = [c for c in CONFIGS if c[0] != "h14final"]
W_SUBJECTS = ALL_SUBJECTS if not SMOKE else ALL_SUBJECTS[:2]
W_SEEDS = [0]                                  # [0, 1, 2] for the final table
if SMOKE: CONFIGS = CONFIGS[:2] + CONFIGS[-1:]

def get_targets(layers, pool):
    tt = [target(l, pool) for l in layers]
    return [a for a, _ in tt], [b for _, b in tt]

for cname, layers, pool, ch in CONFIGS:
    T_tr, T_te = get_targets(layers, pool)
    init = ROUTE_INIT[-len(layers):] if len(layers) > 1 else None
    for seed in W_SEEDS:
        for sub in W_SUBJECTS:
            path = f"{OUT}/within_{cname}_{sub}_s{seed}.json"
            if os.path.exists(path): continue
            t0 = time.time()
            X, Xt = load_eeg(sub, "train", ch), load_eeg(sub, "test", ch)
            rows = torch.arange(N_TR, device=device)
            model, best_ep, best_val = train(X, rows, T_tr, [(VAL, torch.arange(len(VAL), device=device))], seed, init_logits=init)
            sim = embed_eeg(model, Xt) @ embed_img(model, T_te).T
            r = dict(config=cname, layers=[str(l) for l in layers], pool=pool, channels=ch, subject=sub, seed=seed,
                     best_epoch=best_ep, val_top1=best_val, **metrics(sim),
                     route_w=[round(float(w), 3) for w in model.img.weights().detach()], minutes=round((time.time() - t0) / 60, 2))
            save_json(r, path)
            print(f"{cname:11s} {sub} s{seed}: top1 {r['top1']:5.1f} top5 {r['top5']:5.1f} | val {best_val:5.1f} ep {best_ep} "
                  f"| w {r['route_w']} ({r['minutes']} min)", flush=True)
            del X, Xt, model; free_gpu()
    del T_tr, T_te; free_gpu()

W = pd.DataFrame([json.load(open(p)) for p in sorted(glob.glob(f"{OUT}/within_*.json"))])
W.to_csv(f"{OUT}/results_within_v6.csv", index=False)
WS = W.groupby("config")[["top1", "top5", "val_top1"]].agg(["mean", "std"]).round(1)
WS["n"] = W.groupby("config").size()
print(WS.sort_values(("val_top1", "mean"), ascending=False).to_string())

route       sub-01 s0: top1  68.5 top5  90.5 | val  19.2 ep 30 | w [0.07, 0.191, 0.476, 0.192, 0.071] (0.27 min)
route       sub-02 s0: top1  63.0 top5  87.5 | val  13.8 ep 45 | w [0.071, 0.192, 0.473, 0.193, 0.071] (0.23 min)
route       sub-03 s0: top1  61.0 top5  88.0 | val  16.4 ep 25 | w [0.07, 0.19, 0.479, 0.191, 0.07] (0.26 min)
route       sub-04 s0: top1  68.0 top5  91.0 | val  19.2 ep 40 | w [0.071, 0.192, 0.473, 0.193, 0.071] (0.27 min)
route       sub-05 s0: top1  62.5 top5  88.0 | val  10.3 ep 25 | w [0.07, 0.19, 0.478, 0.191, 0.071] (0.26 min)
route       sub-06 s0: top1  72.0 top5  90.5 | val  19.3 ep 40 | w [0.071, 0.191, 0.474, 0.193, 0.071] (0.26 min)
route       sub-07 s0: top1  57.0 top5  86.0 | val  22.3 ep 40 | w [0.071, 0.192, 0.474, 0.193, 0.071] (0.24 min)
route       sub-08 s0: top1  75.0 top5  96.5 | val  26.4 ep 60 | w [0.071, 0.191, 0.474, 0.192, 0.071] (0.21 min)
route       sub-09 s0: top1  61.0 top5  86.5 | val  11.4 ep 25 | w [0.07, 0.19, 0.479, 0.191, 

In [7]:
# 6. Leave-one-subject-out, 10 folds, 63 channels: train on 9 subjects, test on the unseen one (no labels of the new person).
#    Target for LOSO = routing over the layers, plus the best single layer by mean VALIDATION top-1 within-subject.
#    Scoring on the held-out person: plain cosine | CSLS | SCORE-style coordinate recovery (transductive, label-free).
single = W[W.config.str.match(r"^L\d+$")].groupby("config").val_top1.mean()
BEST_SINGLE = single.idxmax() if len(single) else f"L{LAYERS[-3]}"
X_CONFIGS = [("route", LAYERS[-5:]), (BEST_SINGLE, [int(BEST_SINGLE[1:])])]
X_FOLDS = ALL_SUBJECTS if not SMOKE else ALL_SUBJECTS[:2]
print("LOSO targets:", X_CONFIGS)

CSLS_K, N_LAND, RHO = 10, 64, 0.1

@torch.no_grad()
def csls(S, k=CSLS_K):
    rz = S.topk(min(k, S.shape[1]), dim=1).values.mean(1, keepdim=True)
    rt = S.topk(min(k, S.shape[0]), dim=0).values.mean(0, keepdim=True)
    return 2 * S - rz - rt

@torch.no_grad()
def recovery(Z, T, n_land=N_LAND, rho=RHO):
    # SCORE (arXiv 2608.19134) coordinate recovery, one step: moment matching, CSLS, mutual-nearest-neighbour landmarks
    # weighted by their margin, weighted orthogonal Procrustes with identity regularisation, then CSLS scoring.
    Zm = F.normalize((Z - Z.mean(0)) / (Z.std(0) + 1e-6) * T.std(0) + T.mean(0), dim=-1)
    S = csls(Zm @ T.T)
    top2 = S.topk(2, dim=1); j = top2.indices[:, 0]; i = torch.arange(len(S), device=S.device)
    mnn = S.argmax(0)[j] == i
    margin = (top2.values[:, 0] - top2.values[:, 1]).clamp(min=1e-4)
    cand = torch.where(mnn)[0]
    if len(cand) < 3: return S, 0, None
    keep = cand[margin[cand].topk(min(n_land, len(cand))).indices]
    w = margin[keep]; w = len(keep) * w / w.sum()
    Xl, Yl = Zm[keep], T[j[keep]]
    wn = w / w.sum(); muX = (wn[:, None] * Xl).sum(0); muY = (wn[:, None] * Yl).sum(0)
    C = ((Xl - muX) * w[:, None]).T @ (Yl - muY)
    Mx = C + rho * torch.linalg.matrix_norm(C, ord=2) * torch.eye(C.shape[0], device=C.device)
    U, _, Vh = torch.linalg.svd(Mx.double()); R = (U @ Vh).float()
    prec = float((keep == j[keep]).float().mean() * 100)          # diagnostic only (uses the answer key, never used for fitting)
    return csls(F.normalize((Zm - muX) @ R + muY, dim=-1) @ T.T), len(keep), prec

for cname, layers in X_CONFIGS:
    T_tr, T_te = get_targets(layers, "mean")
    init = ROUTE_INIT[-len(layers):] if len(layers) > 1 else None
    for sub in X_FOLDS:
        path = f"{OUT}/loso_{cname}_{sub}.json"
        if os.path.exists(path): print(f"{cname} {sub}: done"); continue
        t0 = time.time(); srcs = [s for s in (ALL_SUBJECTS if not SMOKE else ALL_SUBJECTS[:3]) if s != sub]
        X = torch.empty((len(srcs) * N_TR, N_CH_ALL, N_T), dtype=torch.float16, device=device)
        for k, s in enumerate(srcs): X[k * N_TR:(k + 1) * N_TR] = load_eeg(s, "train", "all")
        img_of = torch.arange(len(X), device=device) % N_TR
        vsets = [(k * N_TR + VAL, torch.arange(len(VAL), device=device)) for k in range(min(3, len(srcs)))]
        log = lambda ep, va, m: print(f"  {cname} {sub} ep {ep:2d} | val top1 {va:.1f} | {(time.time()-t0)/60:.1f} min", flush=True) if ep % 10 == 0 else None
        model, best_ep, best_val = train(X, img_of, T_tr, vsets, 0, log, init)
        del X; free_gpu()
        Xt = load_eeg(sub, "test", "all")
        Z, Ti = embed_eeg(model, Xt), embed_img(model, T_te)
        S0 = Z @ Ti.T; S_rec, n_l, prec = recovery(Z, Ti)
        r = dict(config=cname, test_subject=sub, best_epoch=best_ep, val_top1=best_val,
                 route_w=[round(float(w), 3) for w in model.img.weights().detach()], n_landmarks=n_l, landmark_precision=prec,
                 **{f"plain_{k}": v for k, v in metrics(S0).items()},
                 **{f"csls_{k}": v for k, v in metrics(csls(S0)).items()},
                 **{f"recovery_{k}": v for k, v in metrics(S_rec).items()},
                 minutes=round((time.time() - t0) / 60, 2))
        torch.save(model.state_dict(), f"{OUT}/loso_{cname}_{sub}.pt")
        save_json(r, path)
        print(f"{cname} {sub} ({r['minutes']} min): plain {r['plain_top1']:.1f} | CSLS {r['csls_top1']:.1f} | "
              f"recovery {r['recovery_top1']:.1f} (landmarks {n_l}, precision {prec})", flush=True)
        del Xt, model; free_gpu()
    del T_tr, T_te; free_gpu()

L = pd.DataFrame([json.load(open(p)) for p in sorted(glob.glob(f"{OUT}/loso_*.json"))])
L.to_csv(f"{OUT}/results_loso_v6.csv", index=False)

LOSO targets: [('route', [20, 24, 28, 32, 36]), ('L20', [20])]
  route sub-01 ep 10 | val top1 6.9 | 0.2 min
  route sub-01 ep 20 | val top1 7.2 | 0.3 min
  route sub-01 ep 30 | val top1 6.6 | 0.5 min
  route sub-01 ep 40 | val top1 6.4 | 0.6 min
  route sub-01 ep 50 | val top1 6.4 | 0.7 min
  route sub-01 ep 60 | val top1 6.0 | 0.8 min
route sub-01 (0.83 min): plain 19.5 | CSLS 28.5 | recovery 33.0 (landmarks 64, precision 53.125)
  route sub-02 ep 10 | val top1 7.2 | 0.2 min
  route sub-02 ep 20 | val top1 7.7 | 0.3 min
  route sub-02 ep 30 | val top1 7.1 | 0.5 min
  route sub-02 ep 40 | val top1 6.6 | 0.6 min
  route sub-02 ep 50 | val top1 6.8 | 0.7 min
  route sub-02 ep 60 | val top1 6.7 | 0.8 min
route sub-02 (0.84 min): plain 23.5 | CSLS 34.0 | recovery 39.0 (landmarks 64, precision 70.3125)
  route sub-03 ep 10 | val top1 7.0 | 0.2 min
  route sub-03 ep 20 | val top1 7.0 | 0.3 min
  route sub-03 ep 30 | val top1 7.1 | 0.5 min
  route sub-03 ep 40 | val top1 6.5 | 0.6 min
  rout

In [8]:
# 7. Summary tables (paper numbers are from the papers, under their own protocols: indicative, not like-for-like)
def ms(s): return f"{s.mean():.1f} ± {s.std():.1f} (n={len(s)})"
print("WITHIN-SUBJECT Top-1 (mean ± SD over subjects; seed 0)")
for c, g in W.groupby("config"): print(f"  {c:12s} {ms(g.top1)}   | top5 {g.top5.mean():.1f}")
print("\nLOSO Top-1")
for c, g in L.groupby("config"):
    print(f"  {c:12s} plain {ms(g.plain_top1)} | CSLS {ms(g.csls_top1)} | recovery {ms(g.recovery_top1)}")
print("\nPer subject (LOSO, route):")
print(L[L.config == "route"][["test_subject", "plain_top1", "csls_top1", "recovery_top1", "landmark_precision"]].round(1).to_string(index=False))

REPORTED = pd.DataFrame([
    ["ATM (NeurIPS 2024)", 27.1, 11.9, None], ["UBP (CVPR 2025)", 50.9, 12.4, None], ["NeuroBridge (2026)", 63.2, 19.0, None],
    ["Shallow Alignment (2026)", 82.6, 22.4, None], ["SAMGA (2026)", 91.3, 34.4, None], ["SAMGA + SCORE recovery (2026)", None, 26.2, 48.8],
    ["Ours, notebook 04/05 (old target; LOSO 4 folds)", 28.8, 14.6, None]],
    columns=["method", "intra_top1", "loso_top1", "loso_label_free_top1"])
lr = L[L.config == "route"]
REPORTED.loc[len(REPORTED)] = ["Ours, notebook 06 (route)", round(W[W.config == "route"].top1.mean(), 1),
                               round(lr.plain_top1.mean(), 1), round(lr.recovery_top1.mean(), 1)]
print("\n", REPORTED.to_string(index=False))
REPORTED.to_csv(f"{OUT}/comparison_v6.csv", index=False)

WITHIN-SUBJECT Top-1 (mean ± SD over subjects; seed 0)
  L16          68.5 ± 5.2 (n=10)   | top5 91.9
  L20          70.5 ± 6.0 (n=10)   | top5 92.4
  L24          67.9 ± 6.6 (n=10)   | top5 90.9
  L28          62.7 ± 8.4 (n=10)   | top5 87.7
  L28cls       65.5 ± 5.7 (n=10)   | top5 90.8
  L32          53.7 ± 8.0 (n=10)   | top5 83.1
  L36          48.1 ± 5.4 (n=10)   | top5 78.7
  h14final     32.7 ± 6.8 (n=10)   | top5 63.6
  route        66.3 ± 6.3 (n=10)   | top5 90.0
  route_ch63   50.6 ± 9.6 (n=10)   | top5 81.5

LOSO Top-1
  L20          plain 20.6 ± 6.4 (n=10) | CSLS 30.2 ± 7.4 (n=10) | recovery 34.2 ± 7.1 (n=10)
  route        plain 21.6 ± 5.1 (n=10) | CSLS 31.2 ± 6.2 (n=10) | recovery 35.3 ± 5.5 (n=10)

Per subject (LOSO, route):
test_subject  plain_top1  csls_top1  recovery_top1  landmark_precision
      sub-01        19.5       28.5           33.0                53.1
      sub-02        23.5       34.0           39.0                70.3
      sub-03        14.0       24.5 